In [1]:
# Modules to install
# pip install scikit-clean pandas numpy sklearn

# Importo librerías

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import product

# My libraries
from filters import *
from noisers.classes import *
from noisers.funcs import *
from testFuncs import *
from noise_cv_eval import run_5cv_experiment, run_5cv_grid, run_5cv_baseline

rs = 33

/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dataset_root = Path("dataset")
keel_datasets = sorted([p.name for p in (dataset_root / "data_base").iterdir() if p.is_dir()])
print(f"Available datasets: {keel_datasets}")


Available datasets: ['autos', 'balance', 'banana', 'car', 'cleveland', 'contraceptive', 'dermatology', 'ecoli', 'flare', 'german', 'glass', 'hayes-roth', 'heart', 'ionosphere', 'iris', 'led7digit', 'lymphography', 'magic', 'monk-2', 'newthyroid', 'nursery', 'page-blocks', 'penbased', 'phoneme', 'pima', 'ring', 'satimage', 'segment', 'shuttle', 'sonar', 'spambase', 'splice', 'thyroid', 'twonorm', 'vehicle', 'vowel', 'wdbc', 'wine', 'yeast', 'zoo']


### Display available filters

In [ ]:
print_available_filters()


['ClassificationFilter',
 'ENNFilter',
 'ENNProb',
 'ENNTh',
 'EnsembleFilter',
 'INFFCFilter',
 'IterativePartitioningFilter',
 'MultiEditFilter',
 'NCNEdit',
 'CVCFFilter',
 'TABPFNClassificationFilter']

# Class random noise

In [ ]:
def summarize_results(all_results):
    rows = []
    for res in all_results:
        dataset = res["dataset"]
        baseline_mean = (
            res["baseline_df"][["bal_acc", "f1_macro", "precision_macro", "recall_macro"]]
            .mean()
            .add_prefix("baseline_")
        )
        class_df = res["class_summary_df"].copy()
        class_df["method_base"] = class_df["method"].str.split("_", n=1).str[0]
        best_class = (
            class_df.sort_values(
                by=["method_base", "f1_macro_mean", "bal_acc_mean", "precision_macro_mean", "recall_macro_mean"],
                ascending=[True, False, False, False, False],
            )
            .groupby("method_base", as_index=False)
            .head(1)
        )
        removal_df = res["removal_summary_df"].copy()
        removal_df["method_base"] = removal_df["filter"].str.split("_", n=1).str[0]
        best_removal = (
            removal_df.merge(
                best_class[["dataset", "noise_type", "noise_pct", "seed", "k", "method", "method_base"]],
                left_on=["dataset", "noise_type", "noise_pct", "seed", "k", "method_base"],
                right_on=["dataset", "noise_type", "noise_pct", "seed", "k", "method_base"],
                how="inner",
                suffixes=("", "_bestclass"),
            )
        )
        for _, c_row in best_class.iterrows():
            m = c_row["method_base"]
            r_row = best_removal[best_removal["method_base"] == m].iloc[0] if not best_removal[best_removal["method_base"] == m].empty else None
            row = {
                "dataset": dataset,
                "method_base": m,
                **baseline_mean.to_dict(),
                "class_method": c_row["method"],
                "class_f1_macro_mean": c_row["f1_macro_mean"],
                "class_bal_acc_mean": c_row["bal_acc_mean"],
                "class_precision_macro_mean": c_row["precision_macro_mean"],
                "class_recall_macro_mean": c_row["recall_macro_mean"],
            }
            if r_row is not None:
                row.update({
                    "removal_filter": r_row["filter"],
                    "removal_f1_removal_mean": r_row["f1_removal_mean"],
                    "removal_mcc_mean": r_row["mcc_mean"],
                    "removal_recall_removal_mean": r_row["recall_removal_mean"],
                    "removal_precision_removal_mean": r_row["precision_removal_mean"],
                    "removal_acc_removal_mean": r_row["acc_removal_mean"],
                    "removal_specificity_mean": r_row["specificity_mean"],
                    "removal_removed_pct_mean": r_row["removed_pct_mean"],
                })
            rows.append(row)
    return pd.DataFrame(rows)

## Low noise (5%)

### Filtros basados en distancia

In [ ]:
ENNF

In [ ]:
from sklearn.linear_model import LogisticRegression

# Rejilla global de hiperparámetros
ks = [1, 3, 5, 9, 15]   # Número de vecinos
thresholds = [0.5, 0.75]  # Cota probabilística
n_blocks = [3, 5, 7, 9] 

def build_distance_grid(ks, thresholds, n_blocks):
    '''
    Returns a list of filters, eaach with a different hyperparameter config
    '''
    distance_grid = []
    # ENN base
    for k in ks:
        distance_grid.append({
            "name": f"ENN_k{k}",
            "filter": ENNFilter(n_neighbors=k, mode="enn", njobs=-1),
        })

    # MEEN 
    for k in ks:
        distance_grid.append({
            "name": f"MENN_k{k}",
            "filter": ENNFilter(n_neighbors=k, mode="meen", njobs=-1),
        })

    # ENNProb y ENNProb+Th
    for k in ks:
        distance_grid.append({
            "name": f"ENNProb_k{k}",
            "filter": ENNProbFilter(n_neighbors=k, njobs=-1),
        }) 
        for tau in thresholds:
            distance_grid.append({
                "name": f"ENNTh_k{k}_tau{tau}",
                "filter": ENNProbFilter(n_neighbors=k, mode="th", threshold=tau, njobs=-1),
            })

    #NCNEdit
    for k in ks:
        distance_grid.append({
            "name": f"NCNEdit_k{k}",
            "filter": NCNEdit(n_neighbors=k, njobs=-1),
        })

    # Multiedit
    for k in ks:
        for b in n_blocks:
            distance_grid.append({
                "name": f"MultiEdit_k{k}_b{b}",
                "filter": MultiEditFilter(n_neighbors=k, n_blocks=b, njobs=-1),
            })

    return distance_grid

# Create the distance_based_filter_list
distance_grid = build_distance_grid(ks, thresholds, n_blocks)

# Specify dataset info
noise_type = "cla_rand"
seed = 1
noise_k = 25
folds = (1, 2, 3, 4, 5)

all_results = []

# Test filters in each dataset
for dataset in keel_datasets[:20]:
    # Compute baseline (no filter) results
    baseline_df = run_5cv_baseline(
        dataset=dataset,
        noise_type=noise_type,
        seed=seed,
        k=noise_k,
        encoding="onehot",
        test_source="noisy",
        folds=folds,
    )

    # Compute results using filters
    distance_experiments_5 = [
        {
            "dataset": dataset,
            "noise_type": noise_type,
            "seed": seed,
            "k": noise_k,
            "filters": {cfg["name"]: cfg["filter"]},
            "encoding": "onehot",
            "test_source": "clean",
            "folds": folds,
            "summarize": True,
            "classifier": LogisticRegression(max_iter=1000),
            "experiment_name": f"{dataset}|{noise_type}|{noise_k}|{cfg['name']}",
        }
        for cfg in distance_grid
    ]
    classification_df, removal_df, class_summary_df, removal_summary_df = run_5cv_grid(
        distance_experiments_5,
        save_path="results_5cv",
        save_format="pickle",
    )
    all_results.append(
        {
            "dataset": dataset,
            "baseline_df": baseline_df,
            "classification_df": classification_df,
            "removal_df": removal_df,
            "class_summary_df": class_summary_df,
            "removal_summary_df": removal_summary_df,
        }
    )

5CV experiments:   0%|          | 0/24 [00:00<?, ?it/s]/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
5CV experiments:  17%|█▋        | 4/24 [00:01<00:09,  2.12it/s]/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
5CV experiments:  21%|██        | 5/24 [00:02<00:08,  2.15it/s]/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
5CV experiments:  25%|██▌       | 6/24 [00:02<00:08,  2.12it/s]/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2924: UserWarning:

In [43]:
all_results[0]["baseline_df"].iloc[:,-4:].mean()

bal_acc            0.728556
f1_macro           0.697936
precision_macro    0.761337
recall_macro       0.728556
dtype: float64

In [44]:
all_results[0]["classification_df"].sort_values("bal_acc", ascending=False)

,experiment,dataset,noise_type,noise_pct,seed,k,fold,method,encoding,test_source,preprocess_before_filter,n_train_input,n_train_used,n_test,valid_classification,params,bal_acc,f1_macro,precision_macro,recall_macro
102,autos|cla_rand|25|ENNTh_k9_tau0.5,autos,cla_rand,25,1,25,3,ENNTh_k9_tau0.5,onehot,clean,True,127,40,32,NaN,"{'preprocess_before_filter': True, 'preprocess...",0.564815,0.434815,0.380787,0.564815
57,autos|cla_rand|25|ENNTh_k3_tau0.6,autos,cla_rand,25,1,25,3,ENNTh_k3_tau0.6,onehot,clean,True,127,43,32,NaN,"{'preprocess_before_filter': True, 'preprocess...",0.527778,0.405625,0.359127,0.527778
82,autos|cla_rand|25|ENNTh_k5_tau0.6,autos,cla_rand,25,1,25,3,ENNTh_k5_tau0.6,onehot,clean,True,127,37,32,NaN,"{'preprocess_before_filter': True, 'preprocess...",0.509259,0.390693,0.346154,0.509259
62,autos|cla_rand|25|ENNTh_k3_tau0.7,autos,cla_rand,25,1,25,3,ENNTh_k3_tau0.7,onehot,clean,True,127,36,32,NaN,"{'preprocess_before_filter': True, 'preprocess...",0.509259,0.389752,0.345238,0.509259
98,autos|cla_rand|25|ENNProb_k9,autos,cla_rand,25,1,25,4,ENNProb_k9,onehot,clean,True,127,58,32,NaN,"{'preprocess_before_filter': True, 'preprocess...",0.506667,0.489273,0.550292,0.506667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,autos|cla_rand|25|ENNTh_k3_tau0.8,autos,cla_rand,25,1,25,2,ENNTh_k3_tau0.8,onehot,clean,True,127,28,32,NaN,"{'preprocess_before_filter': True, 'preprocess...",0.228704,0.186147,0.187440,0.228704
91,autos|cla_rand|25|ENNTh_k5_tau0.8,autos,cla_rand,25,1,25,2,ENNTh_k5_tau0.8,onehot,clean,True,127,24,32,NaN,"{'preprocess_before_filter': True, 'preprocess...",0.224074,0.174242,0.160455,0.224074
86,autos|cla_rand|25|ENNTh_k5_tau0.7,autos,cla_rand,25,1,25,2,ENNTh_k5_tau0.7,onehot,clean,True,127,35,32,NaN,"{'preprocess_before_filter': True, 'preprocess...",0.224074,0.167824,0.142256,0.224074
113,autos|cla_rand|25|ENNTh_k9_tau0.7,autos,cla_rand,25,1,25,4,ENNTh_k9_tau0.7,onehot,clean,True,127,25,32,NaN,"{'preprocess_before_filter': True, 'preprocess...",0.220000,0.132549,0.112000,0.220000


In [6]:
df = class_summary_df.copy()
df["method_base"] = df["method"].str.split("_").str[0]
best_by_filter = (
    df.sort_values(
        by=["method_base", "f1_macro_mean", "bal_acc_mean"],
        ascending=[True, False, False],
    )
    .groupby("method_base", as_index=False)
    .head(1)
    .sort_values("f1_macro_mean", ascending=False)
)
best_by_filter

,dataset,noise_type,noise_pct,seed,k,method,params,bal_acc_mean,bal_acc_std,f1_macro_mean,f1_macro_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,method_base
0,cleveland,cla_rand,5,1,5,ENN_k1,"{'preprocess_before_filter': True, 'preprocess...",0.312659,0.042837,0.303493,0.045716,0.315255,0.070011,0.312659,0.042837,ENN
4,cleveland,cla_rand,5,1,5,ENNProb_k1,"{'preprocess_before_filter': True, 'preprocess...",0.312659,0.042837,0.303493,0.045716,0.315255,0.070011,0.312659,0.042837,ENNProb
5,cleveland,cla_rand,5,1,5,ENNTh_k1_tau0.5,"{'preprocess_before_filter': True, 'preprocess...",0.312659,0.042837,0.303493,0.045716,0.315255,0.070011,0.312659,0.042837,ENNTh


In [61]:
df = removal_summary_df.copy()
df["method_base"] = df["filter"].str.split("_").str[0]
best_by_filter_removal = (
    df.sort_values(
        by=["method_base", "f1_removal_mean", "mcc_mean", "recall_removal_mean"],
        ascending=[True, False, False, False],
    )
    .groupby("method_base", as_index=False)
    .head(1)
    .sort_values("f1_removal_mean", ascending=False)
)
best_by_filter_removal

,dataset,noise_type,noise_pct,seed,k,filter,params,removed_pct_mean,removed_pct_std,acc_removal_mean,...,precision_removal_std,recall_removal_mean,recall_removal_std,f1_removal_mean,f1_removal_std,specificity_mean,specificity_std,mcc_mean,mcc_std,method_base
2,pima,cla_rand,50,1,50,ENN_k5,"{'preprocess_before_filter': True, 'preprocess...",43.456529,1.610919,0.653330,...,0.014671,0.688825,0.022644,0.480997,0.016340,0.64255,0.021395,0.282649,0.030614,ENN
13,pima,cla_rand,50,1,50,ENNProb_k5,"{'preprocess_before_filter': True, 'preprocess...",43.814994,1.378762,0.649749,...,0.015135,0.688906,0.032030,0.478340,0.019572,0.63788,0.018399,0.278387,0.035638,ENNProb
14,pima,cla_rand,50,1,50,ENNTh_k5_tau0.5,"{'preprocess_before_filter': True, 'preprocess...",43.814994,1.378762,0.649749,...,0.015135,0.688906,0.032030,0.478340,0.019572,0.63788,0.018399,0.278387,0.035638,ENNTh


### Filtros basados en clasificadores

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

classifier_filters = {
    "ClassificationFilter": ClassificationFilter(estimator=KNeighborsClassifier()),
    "EnsembleFiltering": EnsembleFiltering(
        estimators=[
            GaussianNB(),
            RandomForestClassifier(random_state=rs, n_jobs=-1),
        ],
        cv=5,
        mode="consensus",
        action="remove",
        random_state=rs,
    ),
    "CVCFFilter": CVCFFilter(),
    "IterativePartitioningFilter": IterativePartitioningFilter(),
    "INFFCFilter": INFFCFilter(),
}

classifier_experiments_5 = [
    {
        "dataset": dataset,
        "noise_type": "cla_rand",
        "seed": rs,
        "k": 5,
        "filters": classifier_filters,
        "encoding": "onehot",
        "test_source": "clean",
        "folds": (1, 2, 3, 4, 5),
        "summarize": True,
        "experiment_name": f"{dataset}|cla_rand|5|classifiers",
    }
    for dataset in keel_datasets
]

classifier_class_df_5, classifier_rem_df_5 = run_5cv_grid(classifier_experiments_5)
classifier_class_df_5


## Medium noise (25%)

### Filtros basados en distancia

In [ ]:
distance_experiments_25 = [
    {
        "dataset": dataset,
        "noise_type": "cla_rand",
        "seed": rs,
        "k": 25,
        "filters": distance_filters,
        "encoding": "onehot",
        "test_source": "clean",
        "folds": (1, 2, 3, 4, 5),
        "summarize": True,
        "experiment_name": f"{dataset}|cla_rand|25|distance",
    }
    for dataset in keel_datasets
]

distance_class_df_25, distance_rem_df_25 = run_5cv_grid(distance_experiments_25)
distance_class_df_25


### Filtros basados en clasificadores

In [ ]:
classifier_experiments_25 = [
    {
        "dataset": dataset,
        "noise_type": "cla_rand",
        "seed": rs,
        "k": 25,
        "filters": classifier_filters,
        "encoding": "onehot",
        "test_source": "clean",
        "folds": (1, 2, 3, 4, 5),
        "summarize": True,
        "experiment_name": f"{dataset}|cla_rand|25|classifiers",
    }
    for dataset in keel_datasets
]

classifier_class_df_25, classifier_rem_df_25 = run_5cv_grid(classifier_experiments_25)
classifier_class_df_25


## High noise (50%)

### Filtros basados en distancia

In [ ]:
distance_experiments_50 = [
    {
        "dataset": dataset,
        "noise_type": "cla_rand",
        "seed": rs,
        "k": 50,
        "filters": distance_filters,
        "encoding": "onehot",
        "test_source": "clean",
        "folds": (1, 2, 3, 4, 5),
        "summarize": True,
        "experiment_name": f"{dataset}|cla_rand|50|distance",
    }
    for dataset in keel_datasets
]

distance_class_df_50, distance_rem_df_50 = run_5cv_grid(distance_experiments_50)
distance_class_df_50


### Filtros basados en clasificadores

In [ ]:
classifier_experiments_50 = [
    {
        "dataset": dataset,
        "noise_type": "cla_rand",
        "seed": rs,
        "k": 50,
        "filters": classifier_filters,
        "encoding": "onehot",
        "test_source": "clean",
        "folds": (1, 2, 3, 4, 5),
        "summarize": True,
        "experiment_name": f"{dataset}|cla_rand|50|classifiers",
    }
    for dataset in keel_datasets
]

classifier_class_df_50, classifier_rem_df_50 = run_5cv_grid(classifier_experiments_50)
classifier_class_df_50


## TABPFN (individual)

### Low noise (5%)

In [ ]:
from filters import TABPFNClassificationFilter

tabpfn_classifier_filters = {
    "TABPFNClassificationFilter": TABPFNClassificationFilter(
        cv=5,
        action="remove",
        random_state=rs,
        tabpfn_params={"device": "auto"},
    ),
}

tabpfn_experiments_5 = [
    {
        "dataset": dataset,
        "noise_type": "cla_rand",
        "seed": rs,
        "k": 5,
        "filters": tabpfn_classifier_filters,
        "encoding": "onehot",
        "test_source": "clean",
        "folds": (1, 2, 3, 4, 5),
        "summarize": True,
        "experiment_name": f"{dataset}|cla_rand|5|tabpfn",
    }
    for dataset in keel_datasets
]

tabpfn_class_df_5, tabpfn_rem_df_5 = run_5cv_grid(tabpfn_experiments_5)
tabpfn_class_df_5


### Medium noise (25%)

In [ ]:
tabpfn_experiments_25 = [
    {
        "dataset": dataset,
        "noise_type": "cla_rand",
        "seed": rs,
        "k": 25,
        "filters": tabpfn_classifier_filters,
        "encoding": "onehot",
        "test_source": "clean",
        "folds": (1, 2, 3, 4, 5),
        "summarize": True,
        "experiment_name": f"{dataset}|cla_rand|25|tabpfn",
    }
    for dataset in keel_datasets
]

tabpfn_class_df_25, tabpfn_rem_df_25 = run_5cv_grid(tabpfn_experiments_25)
tabpfn_class_df_25


### High noise (50%)

In [14]:
from filters import TABPFNClassificationFilter
tabpfn_classifier_filters = {
    "TABPFNClassificationFilter": TABPFNClassificationFilter(
        cv=5,
        action="remove",
        random_state=rs,
        tabpfn_params={"device": "auto"},
    ),
}

tabpfn_experiments_50 = [
    {
        "dataset": dataset,
        "noise_type": "cla_rand",
        "seed": 1,
        "k": 50,
        "filters": tabpfn_classifier_filters,
        "encoding": "onehot",
        "test_source": "clean",
        "folds": (1, 2, 3, 4, 5),
        "summarize": True,
        "experiment_name": f"{dataset}|cla_rand|50|tabpfn",
    }
    for dataset in ["iris"]
]

tabpfn_class_df_50, tabpfn_rem_df_50 = run_5cv_grid(tabpfn_experiments_50)
tabpfn_class_df_50


5CV experiments:   0%|          | 0/1 [00:57<?, ?it/s]


KeyboardInterrupt: 